# 🏨 AGODA Price Crawler

Chạy lần lượt các cell từ trên xuống: **① Cấu hình → ② Đọc input → ③ Crawl → ④ Xem kết quả**.

**Đổi nguồn input:** sửa `INPUT_MODE` ở cell ① — `"gsheet"` (Google Sheet online) hoặc `"offline"` (file CSV/XLSX trên máy).

**Format file offline:** chỉ cần **3 cột đầu theo đúng thứ tự** `hotel_name, hotel_url, room_type` (tên cột không quan trọng, chỉ cần đúng thứ tự). Có file mẫu ở `input/TEMPLATE_hotels.csv`.

**Input offline** nằm riêng theo notebook: `input/agoda-thy-7.csv` (cell ① `OFFLINE_FILE` / cell tải Sheet).

**Output** nằm trong `results/agoda/<RUN_NAME>/` — notebook này dùng `thy-7`, không ghi đè notebook khác:
- `FINAL_<YYYYMMDD>.csv` — kết quả cuối
- `TEMP_agoda.csv` — checkpoint: lỡ tắt giữa chừng, chạy lại cell ③ sẽ tự resume phần chưa xong — hotel **chưa từng cào** sẽ chạy trước, hotel còn NA/SOLD OUT retry sau

Cache warm (`results/agoda/captures/`) vẫn dùng chung giữa các notebook nên không tốn thêm thời gian warm.

In [1]:
# ════════════════ ① CẤU HÌNH ════════════════

# ── Tên run: kết quả nằm RIÊNG trong results/agoda/<RUN_NAME>/ ──
RUN_NAME = "thy-7"

# ── Nguồn input: "gsheet" (online) hoặc "offline" (file trên máy) ──
INPUT_MODE = "gsheet"

# Dùng khi INPUT_MODE = "gsheet" (gid của tab được tự lấy từ URL)
GSHEET_URL = "https://docs.google.com/spreadsheets/d/1H_EV1mKY_ID4MiEGsjneL1gvndhq13T29XsyzAmm_jA/edit?gid=1083140588#gid=1083140588"

# Dùng khi INPUT_MODE = "offline" — đường dẫn tuyệt đối, hoặc tương đối so với 31.crawl-tool
# ⚠️ File phải có 3 cột đầu là (tên KS, URL, loại phòng) — file "TEMP_*" là checkpoint OUTPUT, không phải input!
OFFLINE_FILE = "input/agoda-thy-7.csv"

# ── Tham số crawl ──
WEEKS      = 6      # số tuần cần crawl
MAX_HOTELS = 0      # 0 = crawl tất cả; đặt 5 để test nhanh 5 khách sạn đầu
SHARD      = ""     # "" = không chia; "1/3" = chạy phần 1 trong 3 phần (chạy lần lượt 1/3, 2/3, 3/3)

In [2]:
# ════════════════ ② ĐỌC INPUT ════════════════
import os, sys

if "ROOT" not in globals():                    # giữ nguyên ROOT khi chạy lại cell
    ROOT = os.path.abspath("")                 # .../31.crawl-tool (nơi đặt notebook này)
assert os.path.isdir(os.path.join(ROOT, "crawler")), (
    f"Không tìm thấy package `crawler` trong {ROOT} — hãy mở notebook từ thư mục 31.crawl-tool")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import crawler
from crawler.hotels_io import read_hotels

if INPUT_MODE == "gsheet":
    INPUT = GSHEET_URL
    print("📡 Input: Google Sheet online")
else:
    INPUT = OFFLINE_FILE if os.path.isabs(OFFLINE_FILE) else os.path.join(ROOT, OFFLINE_FILE)
    assert os.path.exists(INPUT), f"Không tìm thấy file: {INPUT}"
    print(f"📁 Input: file offline — {INPUT}")

hotels = read_hotels(INPUT)
print(f"✅ Đọc được {len(hotels)} khách sạn. 5 dòng đầu:")
for name, url, room in hotels[:5]:
    print(f"   • {name} — {room}")

📡 Input: Google Sheet online
✅ Đọc được 10 khách sạn. 5 dòng đầu:
   • Vedana Lagoon Wellness Resort & Spa - Villa bên hồ (Water Villa) — Villa bên hồ (Water Villa)
   • Vedana Lagoon Wellness Resort & Spa - Hillside Biệt thự Có hồ bơi (Hillside Pool Villa) — Hillside Biệt thự Có hồ bơi (Hillside Pool Villa)
   • Vedana Lagoon Wellness Resort & Spa - 2 Bedrooms Lagoon View Villa with Free Afternoon Cocktail — 2 Bedrooms Lagoon View Villa with Free Afternoon Cocktail
   • Vedana Lagoon Wellness Resort & Spa - Villa cạnh đồi (Hillside Villa) — Villa cạnh đồi (Hillside Villa)
   • Vedana Lagoon Wellness Resort & Spa - Biệt Thự Hồ Bơi 2 Phòng Ngủ Hướng Đầm Phá - Miễn Phí Cocktail Buổi Chiều (2 Bedrooms Lagoon View Pool Villa - Free afternoon cocktail) — Biệt Thự Hồ Bơi 2 Phòng Ngủ Hướng Đầm Phá - Miễn Phí Cocktail Buổi Chiều (2 Bedrooms Lagoon View Pool Villa - Free afternoon cocktail)


In [3]:
# (TÙY CHỌN) Tải Google Sheet về file offline — lần sau chỉ cần đổi INPUT_MODE = "offline"
import pandas as pd
from crawler.hotels_io import _gsheet_url

os.makedirs(os.path.join(ROOT, "input"), exist_ok=True)
dest = OFFLINE_FILE if os.path.isabs(OFFLINE_FILE) else os.path.join(ROOT, OFFLINE_FILE)
pd.read_csv(_gsheet_url(GSHEET_URL)).to_csv(dest, index=False, encoding="utf-8-sig")
print(f"💾 Đã lưu bản offline: {dest}")

💾 Đã lưu bản offline: /Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/input/agoda-thy-7.csv


In [4]:
# ════════════════ ③ CRAWL ════════════════
OUTDIR = os.path.join(ROOT, "results", "agoda", RUN_NAME)   # mỗi notebook 1 thư mục kết quả riêng
os.makedirs(OUTDIR, exist_ok=True)
os.chdir(OUTDIR)                     # output (FINAL_*.csv, TEMP_agoda.csv) nằm ở đây

kwargs = dict(
    site="agoda",                    # direct replay (nhanh) + Camoufox warm
    input=INPUT,
    weeks=WEEKS,
    # cache warm dùng CHUNG cho mọi notebook (đỡ warm lại) — chỉ kết quả là tách riêng
    capture_dir=os.path.join(ROOT, "results", "agoda", "captures"),
)
if MAX_HOTELS:
    kwargs["max"] = MAX_HOTELS
if SHARD:
    kwargs["shard"] = SHARD

await crawler.arun(**kwargs)         # notebook cho phép await trực tiếp

📂 Resume: 10 rows from TEMP_agoda.csv
🚀 AGODA crawl | 10 hotels × 6w | direct+fallback | engine=camoufox | W1=2026-09-08
🦊 Camoufox ready (humanize=True geoip=True) — browser navs use anti-detect Firefox
✔️  1/10 Vedana Lagoon Wellness Resort & Spa - Villa bên hồ (Water Villa) — complete, skip
✔️  2/10 Vedana Lagoon Wellness Resort & Spa - Hillside Biệt thự Có hồ bơi (Hillside Pool Villa) — complete, skip
✔️  3/10 Vedana Lagoon Wellness Resort & Spa - 2 Bedrooms Lagoon View Villa with Free Afternoon Cocktail — complete, skip
✔️  4/10 Vedana Lagoon Wellness Resort & Spa - Villa cạnh đồi (Hillside Villa) — complete, skip
✔️  5/10 Vedana Lagoon Wellness Resort & Spa - Biệt Thự Hồ Bơi 2 Phòng Ngủ Hướng Đầm Phá - Miễn Phí Cocktail Buổi Chiều (2 Bedrooms Lagoon View Pool Villa - Free afternoon cocktail) — complete, skip
✔️  6/10 Vedana Lagoon Wellness Resort & Spa - Biệt Thự Trên Mặt Nước Có Hồ Bơi - Cocktail Chiều Miễn Phí (Water Pool Villa - Free afternoon cocktail) — complete, skip
✔️  7/

'FINAL_20260903.csv'

In [5]:
# ════════════════ ④ XEM KẾT QUẢ ════════════════
import glob
import pandas as pd

OUTDIR = os.path.join(ROOT, "results", "agoda", RUN_NAME)
files = sorted(glob.glob(os.path.join(OUTDIR, "FINAL_*.csv")))
assert files, "Chưa có file FINAL nào — hãy chạy cell ③ trước."
latest = files[-1]
df = pd.read_csv(latest)
print(f"📄 {latest} — {len(df)} dòng")
df.head(20)

📄 /Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/results/agoda/thy-7/FINAL_20260903.csv — 10 dòng


,hotel_name,room_type,price_w1,price_w2,price_w3,price_w4,price_w5,price_w6
0,Vedana Lagoon Wellness Resort & Spa - Villa bê...,Villa bên hồ (Water Villa),"4,484,618","4,484,618","4,429,943","4,554,914","4,578,437","4,611,080"
1,Vedana Lagoon Wellness Resort & Spa - Hillside...,Hillside Biệt thự Có hồ bơi (Hillside Pool Villa),"4,484,618","4,484,618","5,619,174","5,619,174","4,993,828","4,993,828"
2,Vedana Lagoon Wellness Resort & Spa - 2 Bedroo...,2 Bedrooms Lagoon View Villa with Free Afterno...,"4,984,370","4,984,370","6,245,356","6,245,356","5,550,325","5,550,325"
3,Vedana Lagoon Wellness Resort & Spa - Villa cạ...,Villa cạnh đồi (Hillside Villa),"5,065,849","5,083,396","5,639,225","4,639,511","4,562,475","6,213,366"
4,Vedana Lagoon Wellness Resort & Spa - Biệt Thự...,Biệt Thự Hồ Bơi 2 Phòng Ngủ Hướng Đầm Phá - Mi...,"9,705,562","9,888,557","5,594,354","10,216,793","10,121,353","10,212,637"
5,Vedana Lagoon Wellness Resort & Spa - Biệt Thự...,Biệt Thự Trên Mặt Nước Có Hồ Bơi - Cocktail Ch...,"6,207,448","6,207,448","6,785,560","6,808,696","6,798,947","6,802,792"
6,Ancient Huế Garden Houses (Ancient Hue Garden ...,Phòng Đôi Hướng Vườn (Double Room with Garden ...,"3,406,085","3,406,085","3,406,085","3,406,085","2,934,783","2,934,783"
7,December,Phòng hướng núi 2 giường (Mountain View Twin R...,"1,687,831","1,687,831","3,240,741","1,687,831","1,687,831","1,687,831"
8,Handwritten,"Classic Room - 1 Queen Size Bed, Mountain View","1,330,000","1,330,000","1,260,000","1,260,000","1,260,000","1,260,000"
9,Bellerive,Phòng Loại Sang Hướng Vườn Giường Đôi/Hai Giườ...,"1,322,319","1,524,691","1,524,691","1,524,691","1,335,506","1,335,506"
